# Notebook 05 — Agent Inference Demo

This notebook is the final inference layer for the **Dating Red Flag Detector: Interactive Profile Auditing Agent**.

It loads the artifacts produced by Notebooks 01–04, selects one OkCupid profile, runs the trained NLP and tabular models, attaches the YOLO visual result, combines everything into one structured assessment, and then sends that assessment to OpenRouter for a polished LLM-style report.

Notebook 05 is therefore an **inference/demo notebook**, not another training notebook.

In [1]:
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
!{sys.executable} -m pip install ipywidgets jupyterlab_widgets widgetsnbextension

In [3]:
import ipywidgets as widgets
from IPython.display import display

display(widgets.IntSlider(description="Test"))

IntSlider(value=0, description='Test')

## 1. Imports, paths, and configuration

Run this only after Notebooks 01–04 have produced the cleaned dataset, saved models, visual features, and metric CSVs.

In [4]:
import os
import json
import html
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests

from IPython.display import display, Markdown, HTML, clear_output

warnings.filterwarnings("ignore")

try:
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    TRANSFORMERS_AVAILABLE = True
except Exception as e:
    TRANSFORMERS_AVAILABLE = False
    TRANSFORMER_IMPORT_ERROR = str(e)

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception as e:
    WIDGETS_AVAILABLE = False
    WIDGETS_IMPORT_ERROR = str(e)


def find_project_root():
    # Works whether this notebook is opened from the project root or from a /notebooks folder.
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "data").exists() or (candidate / "models").exists() or (candidate / "reports").exists():
            return candidate
    return Path.cwd()

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

PATHS = {
    "cleaned_data": DATA_DIR / "okcupid_cleaned_redflags.csv",
    "nlp_model": MODELS_DIR / "nlp_tfidf_logreg_model.pkl",
    "nlp_labels": MODELS_DIR / "nlp_label_columns.pkl",
    "nlp_thresholds": MODELS_DIR / "nlp_custom_thresholds.pkl",
    "nlp_label_descriptions": MODELS_DIR / "nlp_label_descriptions.pkl",
    "nlp_keyword_rules": MODELS_DIR / "nlp_keyword_rules.pkl",
    "tabular_models": MODELS_DIR / "tabular_rf_smote_models.pkl",
    "tabular_preprocessor": MODELS_DIR / "tabular_preprocessor.pkl",
    "visual_features": DATA_DIR / "visual_inference_features.csv",
    "visual_metrics": REPORTS_DIR / "yolo_visual_metrics.csv",
    "nlp_metrics": REPORTS_DIR / "nlp_overall_metrics.csv",
    "tabular_metrics": REPORTS_DIR / "tabular_overall_metrics.csv",
}

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Models directory:", MODELS_DIR)
print("Reports directory:", REPORTS_DIR)
print("ipywidgets available:", WIDGETS_AVAILABLE)


Project root: C:\Users\LENOVO\anaconda3\Red-Flagger
Data directory: C:\Users\LENOVO\anaconda3\Red-Flagger\data
Models directory: C:\Users\LENOVO\anaconda3\Red-Flagger\models
Reports directory: C:\Users\LENOVO\anaconda3\Red-Flagger\reports
ipywidgets available: True


## 2. Load dataset, trained models, and previous reports

If this cell raises a missing-file error, rerun the previous notebook that creates that artifact.

In [5]:
required_files = [
    "cleaned_data",
    "nlp_model",
    "nlp_labels",
    "nlp_thresholds",
    "tabular_models",
    "tabular_preprocessor",
]

missing = [name for name in required_files if not PATHS[name].exists()]
if missing:
    raise FileNotFoundError(
        "Missing required files: " + ", ".join(missing) +
        "\nRun Notebooks 01, 02, and 03 before running Notebook 05."
    )

df = pd.read_csv(PATHS["cleaned_data"])

nlp_model = joblib.load(PATHS["nlp_model"])
label_columns = list(joblib.load(PATHS["nlp_labels"]))
nlp_thresholds = joblib.load(PATHS["nlp_thresholds"])

# Optional artifacts produced by the improved Notebook 02.
saved_label_descriptions = joblib.load(PATHS["nlp_label_descriptions"]) if PATHS["nlp_label_descriptions"].exists() else {}
nlp_keyword_rules = joblib.load(PATHS["nlp_keyword_rules"]) if PATHS["nlp_keyword_rules"].exists() else {}

tabular_models = joblib.load(PATHS["tabular_models"])
tabular_preprocessor = joblib.load(PATHS["tabular_preprocessor"])

visual_df = pd.read_csv(PATHS["visual_features"]) if PATHS["visual_features"].exists() else pd.DataFrame()
visual_metrics_df = pd.read_csv(PATHS["visual_metrics"]) if PATHS["visual_metrics"].exists() else pd.DataFrame()
nlp_metrics_df = pd.read_csv(PATHS["nlp_metrics"]) if PATHS["nlp_metrics"].exists() else pd.DataFrame()
tabular_metrics_df = pd.read_csv(PATHS["tabular_metrics"]) if PATHS["tabular_metrics"].exists() else pd.DataFrame()

print("Cleaned dataset shape:", df.shape)
print("NLP labels loaded:", label_columns)
print("Number of NLP labels:", len(label_columns))
print("Optional label descriptions loaded:", bool(saved_label_descriptions))
print("Optional keyword rules loaded:", bool(nlp_keyword_rules))
print("Visual features loaded:", not visual_df.empty)
print("Visual metrics loaded:", not visual_metrics_df.empty)


Cleaned dataset shape: (700, 36)
NLP labels loaded: ['aggressive_tone', 'hookup_focus', 'negativity', 'sarcasm_cynicism', 'substance_risk', 'incomplete_profile', 'controlling_behavior', 'emotional_manipulation', 'entitlement_superiority', 'poor_conflict_resolution', 'main_character_syndrome', 'dismissive']
Number of NLP labels: 12
Optional label descriptions loaded: True
Optional keyword rules loaded: True
Visual features loaded: True
Visual metrics loaded: True


## 3. Label descriptions and profile fields

These make the report easier to understand and reduce how much the LLM has to infer.

In [6]:
# These are the expected labels from the improved Notebook 02.
# If your loaded label_columns only shows the old 6 labels, rerun the improved Notebook 02 first.
expected_label_columns = [
    "aggressive_tone",
    "hookup_focus",
    "negativity",
    "sarcasm_cynicism",
    "substance_risk",
    "incomplete_profile",
    "controlling_behavior",
    "emotional_manipulation",
    "entitlement_superiority",
    "poor_conflict_resolution",
    "main_character_syndrome",
    "dismissive",
]

DEFAULT_LABEL_DESCRIPTIONS = {
    "aggressive_tone": "Harsh, hostile, demanding, or insulting wording.",
    "hookup_focus": "Strong focus on casual hookups or non-committal intent.",
    "negativity": "Pessimistic, bitter, cynical, or emotionally negative statements.",
    "sarcasm_cynicism": "Sarcastic, dismissive, or cynical tone.",
    "substance_risk": "Heavy drinking, smoking, drugs, or party-heavy lifestyle.",
    "incomplete_profile": "Low-effort profile with very little useful information.",
    "controlling_behavior": "Language suggesting control, obedience, punishment, or dominance over a partner.",
    "emotional_manipulation": "Silent treatment, guilt, emotional punishment, testing, or conditional affection.",
    "entitlement_superiority": "Self-important, superior, dismissive, or 'people must impress me' framing.",
    "poor_conflict_resolution": "Cutting people off, refusing communication, revenge, blame, or inability to resolve conflict maturely.",
    "main_character_syndrome": "Self-centred profile framing where the person treats others as background characters or expects special attention.",
    "dismissive": "Invalidating, belittling, or contemptuous wording toward other people's feelings, interests, or boundaries.",
}

# Prefer descriptions saved by Notebook 02, but fill any missing labels from the defaults above.
label_descriptions = DEFAULT_LABEL_DESCRIPTIONS.copy()
if isinstance(saved_label_descriptions, dict):
    label_descriptions.update({k: v for k, v in saved_label_descriptions.items() if v})

missing_expected_labels = [label for label in expected_label_columns if label not in label_columns]
if missing_expected_labels:
    print("Warning: These newer labels are not in the loaded NLP model:", missing_expected_labels)
    print("Rerun the improved Notebook 02 if you want Notebook 05 to detect them.")

visual_flag_descriptions = {
    "clean": "No major visual presentation issue detected.",
    "No_Face_Present": "The image may not show a clear person/face, making identity unclear.",
    "Group_Photo_Ambiguity": "The image contains multiple people, making it unclear who owns the profile.",
    "Face_Obscured": "The face appears hidden, covered, filtered, or visually unclear.",
}

visual_risk_lookup = {
    "clean": 0.0,
    "Face_Obscured": 0.65,
    "Group_Photo_Ambiguity": 0.70,
    "No_Face_Present": 0.80,
}

tabular_features = [
    "age", "status", "sex", "orientation", "body_type", "diet", "drinks", "drugs",
    "education", "ethnicity", "height", "income", "job", "offspring", "pets",
    "religion", "sign", "smokes"
]

profile_display_fields = [
    "age", "status", "sex", "orientation", "body_type", "diet", "drinks", "drugs",
    "education", "job", "offspring", "pets", "religion", "smokes"
]

severe_text_flags = {
    "aggressive_tone",
    "controlling_behavior",
    "emotional_manipulation",
    "entitlement_superiority",
    "poor_conflict_resolution",
    "main_character_syndrome",
    "dismissive",
}

print("Active label descriptions:")
for label in label_columns:
    print(f"- {label}: {label_descriptions.get(label, '[missing description]')}")


Active label descriptions:
- aggressive_tone: Harsh, hostile, demanding, or insulting wording.
- hookup_focus: Strong focus on casual hookups or non-committal intent.
- negativity: Pessimistic, bitter, cynical, or emotionally negative statements.
- sarcasm_cynicism: Sarcastic, dismissive, or cynical tone.
- substance_risk: Heavy drinking, smoking, drugs, or party-heavy lifestyle.
- incomplete_profile: Low-effort profile with very little useful information.
- controlling_behavior: Language suggesting control, obedience, punishment, or dominance over a partner.
- emotional_manipulation: Silent treatment, guilt, emotional punishment, testing, or conditional affection.
- entitlement_superiority: Self-important, superior, dismissive, or 'people must impress me' framing.
- poor_conflict_resolution: Cutting people off, refusing communication, revenge, blame, or inability to resolve conflict maturely.
- main_character_syndrome: Self-centred profile framing where the person treats others as bac

## 4. Select one OkCupid profile

By default, this chooses a random row that has a live YOLO visual result when available. That makes the demo stronger because the visual section can show IoU. Set `PREFER_LIVE_YOLO_SAMPLE = False` for a completely random OkCupid row.

In [7]:
SEED = 42
PREFER_LIVE_YOLO_SAMPLE = True
rng = np.random.default_rng(SEED)

if PREFER_LIVE_YOLO_SAMPLE and not visual_df.empty and "visual_flag_source" in visual_df.columns:
    live_ids = visual_df.loc[visual_df["visual_flag_source"] == "yolo_live", "profile_id"].astype(int).tolist()
    candidate_ids = [idx for idx in live_ids if 0 <= idx < len(df)]
else:
    candidate_ids = list(range(len(df)))

if not candidate_ids:
    candidate_ids = list(range(len(df)))

SAMPLE_PROFILE_ID = int(rng.choice(candidate_ids))
sample_profile = df.iloc[SAMPLE_PROFILE_ID].to_dict()

print("Selected profile_id:", SAMPLE_PROFILE_ID)

display(pd.DataFrame({
    "field": profile_display_fields,
    "value": [sample_profile.get(field, "") for field in profile_display_fields]
}))

bio_preview = str(sample_profile.get("full_bio", ""))[:1500]
display(Markdown("### Bio preview\n" + (bio_preview if bio_preview.strip() else "_No bio text available._")))

Selected profile_id: 0


,field,value
0,age,26
1,status,single
2,sex,m
3,orientation,straight
4,body_type,average
5,diet,anything
6,drinks,socially
7,drugs,never
8,education,graduated from high school
9,job,other


### Bio preview
i am really obsessed with music and would love to do something with it. my music idol is tom delonge. my friends always make fun of me because i buy anything he does. i am supervisor and i am really bored of it. i really need to get back into school. guitar i like to think i am alright. i am always trying to improve at it. well used to be my hair. had to get rid of the blonde mohawk. so now i would say my plugs. i usually get somekind of comment to my earrings. to many movies to just pick one and same with tv shows. but some of the best are scott pilgrim and the office. music is the best thing ever and i surround myself with it everyday. pizza is the best food i could eat it when ever. my brother my guitars my friends my phone my ps3 and oxygen obviously  trying to have fun in what ever is fun to do that night.  want to be friends. to see whats up? or what ever reason you can think of. lets talk

## 5. Inference functions

These functions reuse the saved models from Notebook 02 and Notebook 03, and the exported visual features from Notebook 04.

In [8]:
def _extract_positive_probabilities(predict_proba_output, labels):
    """Normalize sklearn predict_proba outputs into {label: probability}.

    MultiOutputClassifier returns a list of arrays, while some classifiers return a single 2D array.
    This helper keeps Notebook 05 compatible with both older and newer Notebook 02 models.
    """
    probabilities = {}

    if isinstance(predict_proba_output, list):
        for label, arr in zip(labels, predict_proba_output):
            arr = np.asarray(arr)
            if arr.ndim == 2 and arr.shape[1] > 1:
                probabilities[label] = float(arr[0, 1])
            else:
                probabilities[label] = float(arr.ravel()[0])
    else:
        arr = np.asarray(predict_proba_output)
        if arr.ndim == 2 and arr.shape[0] == 1 and arr.shape[1] == len(labels):
            for i, label in enumerate(labels):
                probabilities[label] = float(arr[0, i])
        elif arr.ndim == 2 and arr.shape[1] > 1 and len(labels) == 1:
            probabilities[labels[0]] = float(arr[0, 1])

    return probabilities


def run_nlp_inference(profile_text):
    text = str(profile_text)
    results = {}
    detected_flags = []

    try:
        proba_output = nlp_model.predict_proba([text])
        probability_lookup = _extract_positive_probabilities(proba_output, label_columns)
    except Exception as e:
        return {
            "module": "TF-IDF Logistic Regression NLP model",
            "available": False,
            "reason": f"NLP predict_proba failed: {e}",
            "detected_flags": [],
            "label_scores": {}
        }

    for label in label_columns:
        probability = probability_lookup.get(label, 0.0)
        threshold = float(nlp_thresholds.get(label, 0.5))
        predicted = int(probability >= threshold)

        results[label] = {
            "probability": round(float(probability), 4),
            "threshold": threshold,
            "predicted": predicted,
            "description": label_descriptions.get(label, "")
        }
        if predicted:
            detected_flags.append(label)

    return {
        "module": "TF-IDF Logistic Regression NLP model",
        "available": True,
        "detected_flags": detected_flags,
        "label_scores": results
    }


def run_keyword_rule_inference(profile_text):
    """Optional transparent safety layer from Notebook 02 keyword rules.

    The trained NLP model is still the main detector. This layer helps the demo explain obvious phrases
    such as 'silent treatment', 'I am always right', or 'don't waste my time'.
    """
    if not isinstance(nlp_keyword_rules, dict) or not nlp_keyword_rules:
        return {"available": False, "reason": "No keyword rules artifact found.", "detected_flags": [], "label_scores": {}}

    text = str(profile_text).lower()
    results = {}
    detected_flags = []

    for label in label_columns:
        phrases = nlp_keyword_rules.get(label, [])
        matches = [phrase for phrase in phrases if str(phrase).lower() in text]
        probability = min(0.95, 0.55 + 0.10 * len(matches)) if matches else 0.0
        predicted = int(bool(matches))

        results[label] = {
            "probability": round(float(probability), 4),
            "threshold": 0.55,
            "predicted": predicted,
            "description": label_descriptions.get(label, ""),
            "matched_phrases": matches,
        }
        if predicted:
            detected_flags.append(label)

    return {
        "module": "Transparent keyword/rule safety layer",
        "available": True,
        "detected_flags": detected_flags,
        "label_scores": results,
    }


def run_transformer_inference(profile_text):
    model_path = MODELS_DIR / "distilbert_redflag_model"

    if not TRANSFORMERS_AVAILABLE:
        return {"available": False, "reason": "transformers/torch is not available.", "detected_flags": [], "label_scores": {}}

    if not model_path.exists():
        return {"available": False, "reason": "Saved DistilBERT folder was not found. Rerun Notebook 02 transformer section if needed.", "detected_flags": [], "label_scores": {}}

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSequenceClassification.from_pretrained(model_path)
        model.eval()

        inputs = tokenizer(str(profile_text), return_tensors="pt", truncation=True, padding=True, max_length=256)
        with torch.no_grad():
            outputs = model(**inputs)
            probabilities = torch.sigmoid(outputs.logits).numpy()[0]
    except Exception as e:
        return {"available": False, "reason": f"Transformer inference failed: {e}", "detected_flags": [], "label_scores": {}}

    results = {}
    detected_flags = []
    for label, probability in zip(label_columns, probabilities):
        predicted = int(probability >= 0.5)
        results[label] = {
            "probability": round(float(probability), 4),
            "threshold": 0.5,
            "predicted": predicted,
            "description": label_descriptions.get(label, "")
        }
        if predicted:
            detected_flags.append(label)

    return {"available": True, "module": "DistilBERT Transformer NLP model", "detected_flags": detected_flags, "label_scores": results}


def run_tabular_inference(profile_row):
    input_data = {feature: profile_row.get(feature, np.nan) for feature in tabular_features}
    input_df = pd.DataFrame([input_data])

    try:
        processed_input = tabular_preprocessor.transform(input_df)
    except Exception as e:
        return {"available": False, "reason": f"Tabular preprocessing failed: {e}", "detected_flags": [], "label_scores": {}}

    results = {}
    detected_flags = []

    for label, model in tabular_models.items():
        try:
            probability = model.predict_proba(processed_input)[0][1]
        except Exception:
            probability = 0.0
        predicted = int(probability >= 0.5)
        results[label] = {
            "probability": round(float(probability), 4),
            "threshold": 0.5,
            "predicted": predicted,
            "description": label_descriptions.get(label, "")
        }
        if predicted:
            detected_flags.append(label)

    return {"available": True, "module": "SMOTE Random Forest tabular model", "detected_flags": detected_flags, "label_scores": results}


def safe_json_loads(value, fallback):
    try:
        if pd.isna(value):
            return fallback
        return json.loads(value)
    except Exception:
        return fallback


def normalize_visual_flags(flags):
    if flags is None:
        return ["clean"]
    if isinstance(flags, str):
        flags = [flag.strip() for flag in flags.replace(",", "|").split("|") if flag.strip()]
    flags = list(flags)
    if not flags:
        flags = ["clean"]
    if "clean" in flags and len(flags) > 1:
        flags = [flag for flag in flags if flag != "clean"]
    return flags


def make_manual_visual_assessment(visual_flags="clean", num_persons_detected=1, iou_metric=None, image_path="custom_demo_no_image"):
    flags = normalize_visual_flags(visual_flags)
    visual_risk_score = max([visual_risk_lookup.get(flag, 0.5) for flag in flags] or [0.0])
    return {
        "available": True,
        "module": "Manual visual demo input",
        "detected_visual_flags": flags,
        "visual_risk_score": round(float(visual_risk_score), 4),
        "visual_flag_explanations": {
            flag: visual_flag_descriptions.get(flag, "Custom visual flag entered for demo.")
            for flag in flags
        },
        "bounding_boxes": [],
        "confidence_scores": [],
        "num_persons_detected": int(num_persons_detected),
        "iou_metric": iou_metric,
        "visual_flag_source": "manual_demo_input",
        "image_path": image_path,
    }


def get_visual_assessment(profile_id):
    if visual_df.empty:
        return {"available": False, "reason": "visual_inference_features.csv was not found. Run Notebook 04 first.", "detected_visual_flags": [], "visual_risk_score": 0.0}

    match = visual_df[visual_df["profile_id"].astype(int) == int(profile_id)]
    if match.empty:
        return {"available": False, "reason": f"No visual row found for profile_id {profile_id}.", "detected_visual_flags": [], "visual_risk_score": 0.0}

    row = match.iloc[0].to_dict()
    flags = normalize_visual_flags(row.get("detected_visual_flags", "clean"))
    visual_risk_score = max([visual_risk_lookup.get(flag, 0.5) for flag in flags] or [0.0])

    return {
        "available": True,
        "module": "YOLOv8 visual heuristic module",
        "detected_visual_flags": flags,
        "visual_risk_score": round(float(visual_risk_score), 4),
        "visual_flag_explanations": {flag: visual_flag_descriptions.get(flag, "") for flag in flags},
        "bounding_boxes": safe_json_loads(row.get("bounding_boxes"), []),
        "confidence_scores": safe_json_loads(row.get("confidence_scores"), []),
        "num_persons_detected": int(row.get("num_persons_detected", 0)) if not pd.isna(row.get("num_persons_detected", np.nan)) else 0,
        "iou_metric": None if pd.isna(row.get("iou_metric", np.nan)) else round(float(row.get("iou_metric")), 4),
        "visual_flag_source": row.get("visual_flag_source", "unknown"),
        "image_path": row.get("image_path", None),
    }


## 6. Ensemble auditor

The auditor combines model probabilities into a single risk score and keeps all raw module outputs for explanation.

In [9]:
def combine_label_scores(nlp_result, tabular_result=None, transformer_result=None, keyword_result=None):
    combined = {}

    module_results = [
        ("nlp", nlp_result),
        ("tabular", tabular_result),
        ("transformer", transformer_result),
        ("keyword_rules", keyword_result),
    ]

    for label in label_columns:
        scores = []
        predictions = []
        sources = []
        matched_phrases = []

        for source_name, result in module_results:
            if result and result.get("label_scores") and label in result["label_scores"]:
                label_info = result["label_scores"][label]
                score = float(label_info.get("probability", 0.0))
                predicted = int(label_info.get("predicted", 0))
                scores.append(score)
                predictions.append(predicted)
                if predicted:
                    sources.append(source_name)
                matched_phrases.extend(label_info.get("matched_phrases", []))

        mean_probability = float(np.mean(scores)) if scores else 0.0
        max_probability = float(np.max(scores)) if scores else 0.0
        predicted = int(max_probability >= 0.5 or any(predictions))

        combined[label] = {
            "mean_probability": round(mean_probability, 4),
            "max_probability": round(max_probability, 4),
            "ensemble_predicted": predicted,
            "sources": sources,
            "matched_phrases": sorted(set(matched_phrases)),
            "description": label_descriptions.get(label, "")
        }

    return combined


def risk_level(score):
    if score >= 70:
        return "High"
    if score >= 40:
        return "Moderate"
    if score >= 20:
        return "Low-to-Moderate"
    return "Low"


def calculate_overall_risk(combined_scores, visual_assessment):
    """Convert module output into one cautious profile-audit score.

    The previous version averaged the top probabilities and gave visual input 30% of the score.
    That made very toxic text look too safe when the image was clean. This version is still simple,
    but it is more suitable for a red-flag auditing prototype because severe text flags can push the
    overall score upward even when visual risk is low.
    """
    label_probabilities = [info["max_probability"] for info in combined_scores.values()]
    top_label_risk = float(np.mean(sorted(label_probabilities, reverse=True)[:3])) if label_probabilities else 0.0
    visual_risk = float(visual_assessment.get("visual_risk_score", 0.0))

    severe_detected = [
        label for label, info in combined_scores.items()
        if label in severe_text_flags and info["ensemble_predicted"] == 1
    ]
    severe_boost = min(0.25, 0.06 * len(severe_detected))

    score = (0.75 * top_label_risk + 0.25 * visual_risk + severe_boost) * 100

    # Guardrail: if several severe interpersonal toxicity labels are detected, do not call it low risk.
    if len(severe_detected) >= 3:
        score = max(score, 70.0)
    elif len(severe_detected) >= 2:
        score = max(score, 55.0)

    return round(min(score, 100.0), 2), {
        "top_label_risk": round(top_label_risk, 4),
        "visual_risk": round(visual_risk, 4),
        "severe_detected": severe_detected,
        "severe_boost": round(severe_boost, 4),
    }


def load_metrics_summary():
    summary = {}

    if not nlp_metrics_df.empty:
        summary["nlp_tfidf_logreg_overall"] = nlp_metrics_df.to_dict(orient="records")

    if not tabular_metrics_df.empty:
        summary["tabular_smote_random_forest_overall"] = tabular_metrics_df.to_dict(orient="records")

    if not visual_metrics_df.empty:
        summary["yolo_visual_subset"] = {
            "mean_precision": round(float(visual_metrics_df["precision"].mean()), 4) if "precision" in visual_metrics_df.columns else None,
            "mean_recall": round(float(visual_metrics_df["recall"].mean()), 4) if "recall" in visual_metrics_df.columns else None,
            "mean_iou": round(float(visual_metrics_df["mean_iou"].mean()), 4) if "mean_iou" in visual_metrics_df.columns else None,
            "rows": visual_metrics_df.to_dict(orient="records")
        }

    return summary


def build_profile_assessment(profile_row, profile_id="custom_profile", visual_assessment=None):
    profile_text = profile_row.get("clean_bio", profile_row.get("full_bio", ""))

    nlp = run_nlp_inference(profile_text)
    keyword_rules = run_keyword_rule_inference(profile_text)
    transformer = run_transformer_inference(profile_text)
    tabular = run_tabular_inference(profile_row)

    if visual_assessment is None:
        visual_assessment = {
            "available": False,
            "reason": "No visual input was provided.",
            "detected_visual_flags": [],
            "visual_risk_score": 0.0
        }

    combined_scores = combine_label_scores(nlp, tabular, transformer, keyword_rules)
    detected_text_tabular_flags = [
        label for label, info in combined_scores.items()
        if info["ensemble_predicted"] == 1
    ]

    overall_score, scoring_breakdown = calculate_overall_risk(combined_scores, visual_assessment)

    return {
        "profile_id": profile_id,
        "profile_summary": {field: profile_row.get(field, None) for field in profile_display_fields},
        "bio_excerpt": str(profile_row.get("full_bio", profile_text))[:1200],
        "overall_risk_score": overall_score,
        "overall_risk_level": risk_level(overall_score),
        "scoring_breakdown": scoring_breakdown,
        "detected_text_tabular_flags": detected_text_tabular_flags,
        "detected_visual_flags": visual_assessment.get("detected_visual_flags", []),
        "combined_label_scores": combined_scores,
        "module_outputs": {
            "nlp_tfidf_logreg": nlp,
            "nlp_keyword_rules": keyword_rules,
            "nlp_transformer": transformer,
            "tabular_smote_random_forest": tabular,
            "visual_yolo": visual_assessment,
        },
        "previous_model_performance": load_metrics_summary()
    }


def build_agent_assessment(profile_id):
    profile_row = df.iloc[int(profile_id)].to_dict()
    visual = get_visual_assessment(profile_id)
    return build_profile_assessment(profile_row, profile_id=int(profile_id), visual_assessment=visual)


def build_custom_agent_assessment(custom_profile, visual_assessment=None, profile_id="custom_demo"):
    if visual_assessment is None:
        visual_assessment = make_manual_visual_assessment("clean")
    return build_profile_assessment(custom_profile, profile_id=profile_id, visual_assessment=visual_assessment)


raw_assessment = build_agent_assessment(SAMPLE_PROFILE_ID)
print(json.dumps(raw_assessment, indent=2)[:6000])


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{
  "profile_id": 0,
  "profile_summary": {
    "age": 26,
    "status": "single",
    "sex": "m",
    "orientation": "straight",
    "body_type": "average",
    "diet": "anything",
    "drinks": "socially",
    "drugs": "never",
    "education": "graduated from high school",
    "job": "other",
    "offspring": NaN,
    "pets": NaN,
    "religion": NaN,
    "smokes": "no"
  },
  "bio_excerpt": "i am really obsessed with music and would love to do something with it. my music idol is tom delonge. my friends always make fun of me because i buy anything he does. i am supervisor and i am really bored of it. i really need to get back into school. guitar i like to think i am alright. i am always trying to improve at it. well used to be my hair. had to get rid of the blonde mohawk. so now i would say my plugs. i usually get somekind of comment to my earrings. to many movies to just pick one and same with tv shows. but some of the best are scott pilgrim and the office. music is the best thing 

## 7. Display previous model metrics

This is where Notebook 05 satisfies the assignment requirement to mention Precision, Recall, and YOLO IoU in the final audit report.

In [10]:
if not nlp_metrics_df.empty:
    display(Markdown("### NLP overall metrics"))
    display(nlp_metrics_df)

if not tabular_metrics_df.empty:
    display(Markdown("### Tabular SMOTE overall metrics"))
    display(tabular_metrics_df)

if not visual_metrics_df.empty:
    display(Markdown("### YOLO visual metrics"))
    display(visual_metrics_df)

if nlp_metrics_df.empty and tabular_metrics_df.empty and visual_metrics_df.empty:
    print("No metric CSV files found yet. Rerun Notebooks 02–04 if you want metrics attached to the final report.")

### NLP overall metrics

,Metric,Score
0,Micro Precision,0.477987
1,Micro Recall,0.794425
2,Micro F1,0.596859
3,Macro Precision,0.442378
4,Macro Recall,0.658550
5,Macro F1,0.498826
6,Hamming Loss,0.131624


### Tabular SMOTE overall metrics

,Metric,Score
0,Micro Precision,0.730769
1,Micro Recall,0.339286
2,Micro F1,0.463415
3,Macro Precision,0.116088
4,Macro Recall,0.077570
5,Macro F1,0.092619
6,Hamming Loss,0.078571


### YOLO visual metrics

,image_id,gt_flags,predicted_flags,mean_iou,precision,recall
0,profile_000.jpg,clean,clean,0.437224,0.75,0.75
1,profile_001.jpg,clean,clean,0.576963,0.75,0.75
2,profile_002.jpg,Group_Photo_Ambiguity,Group_Photo_Ambiguity,0.099158,0.75,0.75
3,profile_003.jpg,Group_Photo_Ambiguity,Group_Photo_Ambiguity,0.145029,0.75,0.75
4,profile_004.jpg,Face_Obscured,No_Face_Present,0.000000,0.75,0.75
5,profile_005.jpg,Face_Obscured,No_Face_Present,0.000000,0.75,0.75
6,profile_006.jpg,No_Face_Present,No_Face_Present,1.000000,0.75,0.75
7,profile_007.jpg,No_Face_Present,No_Face_Present,1.000000,0.75,0.75


## 8. OpenRouter beautification layer

This sends the structured assessment to OpenRouter and asks an LLM to rewrite it into a polished report.

Do **not** hardcode your API key in the submitted notebook. Set it as an environment variable instead:

```python
os.environ["OPENROUTER_API_KEY"] = "your_key_here"
os.environ["OPENROUTER_MODEL"] = "openrouter/auto"
```

If no API key is set, the notebook uses a local fallback report so that the demo still runs.

In [34]:
def call_openrouter_for_report(assessment, model="openrouter/free"):
    import os
    import json
    import requests

    api_key = os.getenv("OPENROUTER_API_KEY")

    if api_key is None or api_key.strip() == "":
        raise RuntimeError("OPENROUTER_API_KEY is not set")

    # Clean accidental spaces or quote marks
    api_key = api_key.strip().strip('"').strip("'")

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",
        "X-OpenRouter-Title": "Dating Red Flag Detector Demo"
    }

    prompt = f"""
You are the reporting layer for a dating profile red-flag auditing prototype.

Your job:
- Do not invent new model findings.
- Rewrite the model assessment into a clear, readable audit report.
- Keep the tone careful and non-defamatory.
- Explain that this is a prototype decision-support tool.
- Mention detected labels, confidence values, visual findings, and model limitations.

Raw assessment JSON:
{json.dumps(assessment, indent=2)}
"""

    payload = {
        "model": model,
        "messages": [
            {
                "role": "system",
                "content": "You convert model outputs into careful dating profile audit reports."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0.3
    }

    print("OpenRouter key found:", bool(api_key))
    print("Authorization header starts with:", headers["Authorization"][:18])

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=headers,
        json=payload,
        timeout=60
    )

    if response.status_code != 200:
        print("Status:", response.status_code)
        print("Raw response:", response.text)
        response.raise_for_status()

    data = response.json()
    return data["choices"][0]["message"]["content"]

## 9. Local fallback report

This produces a report without using the API. It is useful for testing or for submission screenshots when the API is unavailable.

In [12]:
def _format_metric_rows(rows):
    formatted = []
    for row in rows:
        metric = row.get("Metric", row.get("metric", "Metric"))
        score = row.get("Score", row.get("score", None))
        if isinstance(score, (int, float, np.floating)):
            formatted.append(f"{metric}: {float(score):.4f}")
        else:
            formatted.append(f"{metric}: {score}")
    return ", ".join(formatted)


def generate_local_report(assessment):
    risk_score = assessment["overall_risk_score"]
    risk_level_text = assessment["overall_risk_level"]
    text_flags = assessment["detected_text_tabular_flags"]
    visual_flags = assessment["detected_visual_flags"]
    combined_scores = assessment["combined_label_scores"]
    visual = assessment["module_outputs"]["visual_yolo"]
    scoring = assessment.get("scoring_breakdown", {})

    lines = []
    lines.append(f"# Dating Profile Audit Report — Profile {assessment['profile_id']}")
    lines.append("")
    lines.append(f"**Overall result:** {risk_level_text} risk ({risk_score}/100).")
    lines.append("")
    lines.append("This is an automated prototype assessment. It should support user decision-making, not judge the person as inherently unsafe.")
    lines.append("")

    lines.append("## 1. Text and bio red flags")
    if text_flags:
        for flag in text_flags:
            info = combined_scores[flag]
            source_text = f" Sources: {', '.join(info.get('sources', []))}." if info.get("sources") else ""
            matched = info.get("matched_phrases", [])
            match_text = f" Matched phrases: {', '.join(matched[:4])}." if matched else ""
            lines.append(
                f"- **{flag}** — {info['description']} "
                f"Max probability: {info['max_probability']:.4f}; mean ensemble probability: {info['mean_probability']:.4f}."
                f"{source_text}{match_text}"
            )
    else:
        lines.append("- No major text/tabular red flags were detected by the ensemble threshold.")
    lines.append("")

    lines.append("## 2. Tabular/profile metadata signals")
    tabular_output = assessment["module_outputs"].get("tabular_smote_random_forest", {})
    if tabular_output.get("detected_flags"):
        for flag in tabular_output["detected_flags"]:
            score = tabular_output["label_scores"][flag]["probability"]
            lines.append(f"- **{flag}** was triggered by the tabular SMOTE model with probability {score:.4f}.")
    elif tabular_output.get("available") is False:
        lines.append(f"- Tabular model unavailable: {tabular_output.get('reason')}")
    else:
        lines.append("- The tabular SMOTE model did not independently trigger any metadata flag.")
    lines.append("")

    lines.append("## 3. Visual profile image signals")
    if visual.get("available"):
        for flag in visual_flags:
            explanation = visual.get("visual_flag_explanations", {}).get(flag, "")
            lines.append(f"- **{flag}** — {explanation}")
        lines.append(f"- Detected persons: {visual.get('num_persons_detected')}")
        lines.append(f"- IoU for this row: {visual.get('iou_metric')}")
        lines.append(f"- Visual source: {visual.get('visual_flag_source')}")
    else:
        lines.append(f"- Visual module unavailable: {visual.get('reason')}")
    lines.append("")

    lines.append("## 4. Scoring explanation")
    if scoring:
        severe_detected = scoring.get("severe_detected", [])
        lines.append(f"- Text risk from strongest labels: {scoring.get('top_label_risk')}.")
        lines.append(f"- Visual risk contribution: {scoring.get('visual_risk')}.")
        lines.append(f"- Severe interpersonal toxicity labels detected: {', '.join(severe_detected) if severe_detected else 'None'}.")
        lines.append(f"- Severe-label caution boost: {scoring.get('severe_boost')}.")
    else:
        lines.append("- No scoring breakdown was attached.")
    lines.append("")

    lines.append("## 5. Model performance note")
    perf = assessment.get("previous_model_performance", {})

    if "nlp_tfidf_logreg_overall" in perf:
        lines.append(f"- NLP model overall metrics: {_format_metric_rows(perf['nlp_tfidf_logreg_overall'])}.")

    if "tabular_smote_random_forest_overall" in perf:
        lines.append(f"- Tabular SMOTE model overall metrics: {_format_metric_rows(perf['tabular_smote_random_forest_overall'])}.")

    if "yolo_visual_subset" in perf:
        yolo = perf["yolo_visual_subset"]
        lines.append(f"- YOLO visual subset metrics: Precision {yolo.get('mean_precision')}, Recall {yolo.get('mean_recall')}, Mean IoU {yolo.get('mean_iou')}.")

    if not perf:
        lines.append("- Metric CSV files were not found, so performance metrics were not attached.")
    lines.append("")

    lines.append("## 6. Final recommendation")
    if risk_score >= 70:
        lines.append("- The app should warn the user to review this profile carefully before engaging.")
    elif risk_score >= 40:
        lines.append("- The app should show a caution note and highlight the specific evidence.")
    else:
        lines.append("- The app can mark the profile as relatively low-risk while still showing transparent model evidence.")

    return "\n".join(lines)


## 10. Generate and save default sample report

This runs the auditor on the selected OkCupid row. The custom interface comes after this section.

In [13]:
try:
    llm_report = call_openrouter_for_report(raw_assessment)
except Exception as e:
    print("OpenRouter call failed. Using local fallback report instead.")
    print("Reason:", e)
    llm_report = None

final_report = llm_report if llm_report else generate_local_report(raw_assessment)

display(Markdown(final_report))

assessment_path = REPORTS_DIR / "agent_sample_assessment.json"
report_path = REPORTS_DIR / "agent_sample_report.md"

with open(assessment_path, "w", encoding="utf-8") as f:
    json.dump(raw_assessment, f, indent=2)

with open(report_path, "w", encoding="utf-8") as f:
    f.write(final_report)

print("Saved raw assessment to:", assessment_path)
print("Saved final report to:", report_path)


# Dating Profile Audit Report — Profile 0

**Overall result:** Moderate risk (45.06/100).

This is an automated prototype assessment. It should support user decision-making, not judge the person as inherently unsafe.

## 1. Text and bio red flags
- **sarcasm_cynicism** — Sarcastic, dismissive, or cynical tone. Max probability: 0.9191; mean ensemble probability: 0.5277. Sources: nlp, tabular.

## 2. Tabular/profile metadata signals
- **sarcasm_cynicism** was triggered by the tabular SMOTE model with probability 0.8000.

## 3. Visual profile image signals
- **clean** — No major visual presentation issue detected.
- Detected persons: 1
- IoU for this row: 0.4372
- Visual source: yolo_live

## 4. Scoring explanation
- Text risk from strongest labels: 0.6008.
- Visual risk contribution: 0.0.
- Severe interpersonal toxicity labels detected: None.
- Severe-label caution boost: 0.0.

## 5. Model performance note
- NLP model overall metrics: Micro Precision: 0.4780, Micro Recall: 0.7944, Micro F1: 0.5969, Macro Precision: 0.4424, Macro Recall: 0.6586, Macro F1: 0.4988, Hamming Loss: 0.1316.
- Tabular SMOTE model overall metrics: Micro Precision: 0.7308, Micro Recall: 0.3393, Micro F1: 0.4634, Macro Precision: 0.1161, Macro Recall: 0.0776, Macro F1: 0.0926, Hamming Loss: 0.0786.
- YOLO visual subset metrics: Precision 0.75, Recall 0.75, Mean IoU 0.4073.

## 6. Final recommendation
- The app should show a caution note and highlight the specific evidence.

Saved raw assessment to: C:\Users\LENOVO\anaconda3\Red-Flagger\reports\agent_sample_assessment.json
Saved final report to: C:\Users\LENOVO\anaconda3\Red-Flagger\reports\agent_sample_report.md


## 11. Conceptual mobile interface mockup

This is not a real app yet. It is a visual prototype of how the auditor result could appear in the final mobile interface deliverable.

In [14]:
def render_mobile_auditor_card(assessment):
    risk_score = assessment["overall_risk_score"]
    risk_level_text = assessment["overall_risk_level"]
    text_flags = assessment["detected_text_tabular_flags"]
    visual_flags = assessment["detected_visual_flags"]

    flag_html = "".join(
        f"<span class='pill'>{html.escape(flag)}</span>"
        for flag in (text_flags + visual_flags)
    ) or "<span class='pill clean'>No major flags</span>"

    bio_excerpt = html.escape(str(assessment.get("bio_excerpt", ""))[:350])

    if risk_score >= 70:
        risk_class = "high"
    elif risk_score >= 40:
        risk_class = "moderate"
    else:
        risk_class = "low"

    card = f"""
    <style>
        .phone-frame {{
            width: 360px;
            border-radius: 32px;
            padding: 18px;
            background: #111827;
            color: #f9fafb;
            font-family: Arial, sans-serif;
            box-shadow: 0 12px 30px rgba(0,0,0,0.25);
        }}
        .phone-card {{
            background: #1f2937;
            border-radius: 24px;
            padding: 18px;
            border: 1px solid #374151;
        }}
        .small {{ color: #9ca3af; font-size: 12px; }}
        .score {{ font-size: 42px; font-weight: bold; margin: 8px 0; }}
        .level {{ font-size: 16px; margin-bottom: 12px; }}
        .pill {{
            display: inline-block;
            padding: 6px 10px;
            border-radius: 999px;
            margin: 4px 4px 4px 0;
            background: #374151;
            font-size: 12px;
        }}
        .clean {{ background: #065f46; }}
        .low {{ color: #34d399; }}
        .moderate {{ color: #fbbf24; }}
        .high {{ color: #f87171; }}
        .section {{ margin-top: 16px; }}
    </style>
    <div class="phone-frame">
        <div class="phone-card">
            <div class="small">Dating Red Flag Detector</div>
            <h2>Profile Audit</h2>
            <div class="small">Profile ID: {html.escape(str(assessment['profile_id']))}</div>
            <div class="score {risk_class}">{risk_score}</div>
            <div class="level">{html.escape(risk_level_text)} Risk</div>
            <div class="section">
                <div class="small">Detected Signals</div>
                {flag_html}
            </div>
            <div class="section">
                <div class="small">Bio Preview</div>
                <p>{bio_excerpt}...</p>
            </div>
            <div class="section small">
                Tap to view model evidence, Precision/Recall, and YOLO IoU details.
            </div>
        </div>
    </div>
    """
    return HTML(card)


display(render_mobile_auditor_card(raw_assessment))


## 12. Custom profile input interface

Use this section to type your own demo dating profile and immediately generate the audit result. This is useful for presentation because it shows the agent working interactively instead of only using a random dataset row.

In [35]:
import os

os.environ["OPENROUTER_API_KEY"] = "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"
api_key = os.getenv("OPENROUTER_API_KEY", "").strip()
print("Key exists:", bool(api_key))
print("Key starts with:", api_key[:10])
print("Key length:", len(api_key))

Key exists: True
Key starts with: sk-or-v1-b
Key length: 73


In [36]:
def make_custom_profile_from_inputs(
    bio,
    age=27,
    status="single",
    sex="m",
    orientation="straight",
    body_type="average",
    diet="mostly anything",
    drinks="socially",
    drugs="never",
    education="graduated from college/university",
    ethnicity="asian",
    height=170,
    income=-1,
    job="student",
    offspring="doesn’t have kids",
    pets="likes dogs",
    religion="agnosticism",
    sign="leo",
    smokes="no",
):
    return {
        "full_bio": str(bio),
        "clean_bio": str(bio),
        "age": age,
        "status": status,
        "sex": sex,
        "orientation": orientation,
        "body_type": body_type,
        "diet": diet,
        "drinks": drinks,
        "drugs": drugs,
        "education": education,
        "ethnicity": ethnicity,
        "height": height,
        "income": income,
        "job": job,
        "offspring": offspring,
        "pets": pets,
        "religion": religion,
        "sign": sign,
        "smokes": smokes,
    }


def audit_custom_profile(custom_profile, visual_flags="clean", profile_id="custom_demo_001", use_openrouter=False, save_prefix="custom_demo"):
    visual_assessment = make_manual_visual_assessment(visual_flags=visual_flags)
    assessment = build_custom_agent_assessment(
        custom_profile=custom_profile,
        visual_assessment=visual_assessment,
        profile_id=profile_id,
    )

    if use_openrouter:
        try:
            llm_report = call_openrouter_for_report(assessment)
        except Exception as e:
            print("OpenRouter call failed. Using local fallback report instead.")
            print("Reason:", e)
            llm_report = None
    else:
        llm_report = None

    report = llm_report if llm_report else generate_local_report(assessment)

    assessment_path = REPORTS_DIR / f"{save_prefix}_assessment.json"
    report_path = REPORTS_DIR / f"{save_prefix}_report.md"

    with open(assessment_path, "w", encoding="utf-8") as f:
        json.dump(assessment, f, indent=2)

    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report)

    return assessment, report, assessment_path, report_path


if not WIDGETS_AVAILABLE:
    print("ipywidgets is not available, so the interactive form cannot be displayed.")
    print("Use the fallback custom_profile code cell below instead.")
else:
    bio_box = widgets.Textarea(
        value=(
            "I'm not like everyone else on here. I am always right. "
            "Silent treatment is a must for the other party to learn from their mistakes. "
            "People should already know what I want if they care enough."
        ),
        description="Bio",
        layout=widgets.Layout(width="100%", height="160px"),
    )

    profile_id_box = widgets.Text(value="custom_demo_001", description="Profile ID")
    age_slider = widgets.IntSlider(value=27, min=18, max=80, step=1, description="Age")
    height_slider = widgets.IntSlider(value=170, min=120, max=220, step=1, description="Height")
    income_box = widgets.IntText(value=-1, description="Income")

    status_dropdown = widgets.Dropdown(options=["single", "seeing someone", "available", "married"], value="single", description="Status")
    sex_dropdown = widgets.Dropdown(options=["m", "f"], value="m", description="Sex")
    orientation_dropdown = widgets.Dropdown(options=["straight", "gay", "bisexual"], value="straight", description="Orientation")
    body_type_dropdown = widgets.Dropdown(options=["average", "fit", "athletic", "thin", "curvy", "a little extra", "overweight", "rather not say"], value="average", description="Body")
    diet_dropdown = widgets.Dropdown(options=["mostly anything", "anything", "vegetarian", "vegan", "halal", "kosher", "other"], value="mostly anything", description="Diet")
    drinks_dropdown = widgets.Dropdown(options=["not at all", "rarely", "socially", "often", "very often", "desperately"], value="socially", description="Drinks")
    drugs_dropdown = widgets.Dropdown(options=["never", "sometimes", "often"], value="never", description="Drugs")
    smokes_dropdown = widgets.Dropdown(options=["no", "sometimes", "when drinking", "yes", "trying to quit"], value="no", description="Smokes")
    education_dropdown = widgets.Dropdown(options=["graduated from high school", "graduated from college/university", "working on college/university", "dropped out of college/university", "graduated from masters program", "other"], value="graduated from college/university", description="Education")
    ethnicity_text = widgets.Text(value="asian", description="Ethnicity")
    job_text = widgets.Text(value="student", description="Job")
    offspring_dropdown = widgets.Dropdown(options=["doesn’t have kids", "has kids", "had kids", "doesn’t want kids", "wants kids", "might want kids"], value="doesn’t have kids", description="Kids")
    pets_dropdown = widgets.Dropdown(options=["likes dogs", "likes cats", "likes dogs and cats", "has dogs", "has cats", "dislikes pets"], value="likes dogs", description="Pets")
    religion_text = widgets.Text(value="agnosticism", description="Religion")
    sign_text = widgets.Text(value="leo", description="Sign")

    visual_select = widgets.SelectMultiple(
        options=["clean", "No_Face_Present", "Group_Photo_Ambiguity", "Face_Obscured"],
        value=("clean",),
        description="Visual",
        layout=widgets.Layout(height="90px"),
    )
    openrouter_toggle = widgets.Checkbox(value=False, description="Use OpenRouter beautification")
    run_button = widgets.Button(description="Audit Custom Profile", button_style="danger")
    output_area = widgets.Output()

    def on_audit_button_clicked(_):
        with output_area:
            clear_output()
            profile = make_custom_profile_from_inputs(
                bio=bio_box.value,
                age=age_slider.value,
                status=status_dropdown.value,
                sex=sex_dropdown.value,
                orientation=orientation_dropdown.value,
                body_type=body_type_dropdown.value,
                diet=diet_dropdown.value,
                drinks=drinks_dropdown.value,
                drugs=drugs_dropdown.value,
                education=education_dropdown.value,
                ethnicity=ethnicity_text.value,
                height=height_slider.value,
                income=income_box.value,
                job=job_text.value,
                offspring=offspring_dropdown.value,
                pets=pets_dropdown.value,
                religion=religion_text.value,
                sign=sign_text.value,
                smokes=smokes_dropdown.value,
            )
            visual_flags = list(visual_select.value)
            assessment, report, assessment_path, report_path = audit_custom_profile(
                custom_profile=profile,
                visual_flags=visual_flags,
                profile_id=profile_id_box.value,
                use_openrouter=openrouter_toggle.value,
                save_prefix=str(profile_id_box.value).replace(" ", "_")
            )

            display(Markdown(report))
            display(render_mobile_auditor_card(assessment))

            score_rows = []
            for label, info in assessment["combined_label_scores"].items():
                score_rows.append({
                    "label": label,
                    "predicted": info["ensemble_predicted"],
                    "max_probability": info["max_probability"],
                    "mean_probability": info["mean_probability"],
                    "sources": ", ".join(info.get("sources", [])),
                    "matched_phrases": ", ".join(info.get("matched_phrases", [])[:5]),
                })
            score_df = pd.DataFrame(score_rows).sort_values(["predicted", "max_probability"], ascending=[False, False])
            display(Markdown("### Label evidence table"))
            display(score_df)

            print("Saved custom raw assessment to:", assessment_path)
            print("Saved custom final report to:", report_path)

    run_button.on_click(on_audit_button_clicked)

    left_col = widgets.VBox([
        profile_id_box,
        bio_box,
        widgets.HBox([age_slider, height_slider]),
        widgets.HBox([status_dropdown, sex_dropdown, orientation_dropdown]),
        widgets.HBox([body_type_dropdown, diet_dropdown]),
        widgets.HBox([drinks_dropdown, drugs_dropdown, smokes_dropdown]),
        widgets.HBox([education_dropdown, job_text]),
        widgets.HBox([offspring_dropdown, pets_dropdown]),
        widgets.HBox([ethnicity_text, religion_text, sign_text]),
        visual_select,
        openrouter_toggle,
        run_button,
    ])

    display(left_col, output_area)


Output()

### Fallback custom profile cell

Use this if `ipywidgets` is unavailable, or if you prefer editing a plain Python dictionary.

In [16]:
# Plain-code fallback demo. Edit the fields, then run this cell.
custom_profile = make_custom_profile_from_inputs(
    bio="""
    I'm not like everyone else on here. I am always right. Silent treatment is a must
    for the other party to learn from their mistakes. People should know what I want
    without me explaining it. If someone wrongs me, I cut them off because they no longer
    deserve access to me.
    """,
    age=30,
    status="single",
    sex="m",
    orientation="straight",
    body_type="average",
    diet="mostly anything",
    drinks="often",
    drugs="sometimes",
    education="graduated from college/university",
    ethnicity="asian",
    height=170,
    income=-1,
    job="CEO",
    offspring="doesn’t have kids",
    pets="likes dogs",
    religion="agnosticism",
    sign="leo",
    smokes="sometimes",
)

custom_assessment, custom_report, custom_assessment_path, custom_report_path = audit_custom_profile(
    custom_profile=custom_profile,
    visual_flags=["clean"],
    profile_id="custom_demo_fallback",
    use_openrouter=False,
    save_prefix="custom_demo_fallback",
)

display(Markdown(custom_report))
display(render_mobile_auditor_card(custom_assessment))

print("Saved custom raw assessment to:", custom_assessment_path)
print("Saved custom final report to:", custom_report_path)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

# Dating Profile Audit Report — Profile custom_demo_fallback

**Overall result:** High risk (84.94/100).

This is an automated prototype assessment. It should support user decision-making, not judge the person as inherently unsafe.

## 1. Text and bio red flags
- **substance_risk** — Heavy drinking, smoking, drugs, or party-heavy lifestyle. Max probability: 0.9100; mean ensemble probability: 0.3571. Sources: tabular, transformer.
- **emotional_manipulation** — Silent treatment, guilt, emotional punishment, testing, or conditional affection. Max probability: 0.9161; mean ensemble probability: 0.3054. Sources: nlp.
- **entitlement_superiority** — Self-important, superior, dismissive, or 'people must impress me' framing. Max probability: 0.4795; mean ensemble probability: 0.1632. Sources: nlp.
- **poor_conflict_resolution** — Cutting people off, refusing communication, revenge, blame, or inability to resolve conflict maturely. Max probability: 0.8515; mean ensemble probability: 0.2855. Sources: nlp.

## 2. Tabular/profile metadata signals
- **substance_risk** was triggered by the tabular SMOTE model with probability 0.9100.

## 3. Visual profile image signals
- **clean** — No major visual presentation issue detected.
- Detected persons: 1
- IoU for this row: None
- Visual source: manual_demo_input

## 4. Scoring explanation
- Text risk from strongest labels: 0.8925.
- Visual risk contribution: 0.0.
- Severe interpersonal toxicity labels detected: emotional_manipulation, entitlement_superiority, poor_conflict_resolution.
- Severe-label caution boost: 0.18.

## 5. Model performance note
- NLP model overall metrics: Micro Precision: 0.4780, Micro Recall: 0.7944, Micro F1: 0.5969, Macro Precision: 0.4424, Macro Recall: 0.6586, Macro F1: 0.4988, Hamming Loss: 0.1316.
- Tabular SMOTE model overall metrics: Micro Precision: 0.7308, Micro Recall: 0.3393, Micro F1: 0.4634, Macro Precision: 0.1161, Macro Recall: 0.0776, Macro F1: 0.0926, Hamming Loss: 0.0786.
- YOLO visual subset metrics: Precision 0.75, Recall 0.75, Mean IoU 0.4073.

## 6. Final recommendation
- The app should warn the user to review this profile carefully before engaging.

Saved custom raw assessment to: C:\Users\LENOVO\anaconda3\Red-Flagger\reports\custom_demo_fallback_assessment.json
Saved custom final report to: C:\Users\LENOVO\anaconda3\Red-Flagger\reports\custom_demo_fallback_report.md
